# 12 Сквозная диагностика pi0 strict-input

        Этот раздел разделяет четыре вещи, которые при отладке pi0 легче всего спутать: raw policy; learned head, использующий только допустимые для VLA входы; стабильность сэмплирования политики в фиксированной сцене и настоящая генерализация на случайные положения объектов. Все таблицы основаны на измерениях на устройстве AMD.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("Запустите Jupyter из каталога темы AMD ROCm или из подкаталога notebooks.")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


TOPIC_ROOT = $NOTEBOOK_TOPIC_ROOT
PROJECT_ROOT = $PROJECT_ROOT
DATA_ROOT = $PROJECT_ROOT
OUTPUT_ROOT = $OUTPUT_ROOT
MODEL_ROOT = $MODEL_ROOT


In [2]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Нет файла материала: {path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1: чтение итоговых строгих результатов оценки


In [3]:
results = json.loads(
    (ASSET_DIR / "pi0_strict_input_results.json").read_text(encoding="utf-8")
)
fixed = results["fixed_scene"]
randomized = results["randomized_environment_gate"]
rows = [
    (
        "raw pi0",
        "env seed 0; 12 policy sampling seeds",
        f'{fixed["raw_pi0"]["success"]}/{fixed["raw_pi0"]["total"]}',
    ),
    (
        "pi0 + visual/history GRU head",
        "env seed 0; та же партия из 12 policy sampling seeds",
        f'{fixed["pi0_visual_history_gru"]["success"]}/{fixed["pi0_visual_history_gru"]["total"]}',
    ),
    (
        "pi0 + visual/history GRU head",
        "случайные окружения 41–44 после исправления seed",
        f'{randomized["pi0_visual_history_gru"]["success"]}/{randomized["pi0_visual_history_gru"]["total"]}',
    ),
]
md_table(["Стратегия", "Протокол", "final strict"], rows)


| 策略 | 协议 | final strict |
| --- | --- | --- |
| raw pi0 | env seed 0；12 个 policy sampling seeds | 0/12 |
| pi0 + visual/history GRU head | env seed 0；同一批 12 个 policy sampling seeds | 6/12 |
| pi0 + visual/history GRU head | 修正 seed 后的随机环境 41–44 | 1/4 |


![Сравнение pi0 strict-input](../assets/pi0_strict_input_progress.png)

        Рост в фиксированной сцене с `0/12` до `6/12` — существенный прогресс, но нельзя пропускать случайные окружения `1/4`. Именно поэтому учебник отдельно описывает «подгонку под фиксированную сцену» и «генерализацию по положению».


## Checkpoint 2: что на самом деле видит learned head


In [4]:
rows = [
    ("Разрешено", "изображения agent/wrist, язык, 6D robot proprio, 7D команды EEF/gripper, которые предыдущая политика выполнила сама"),
    ("Запрещено", "target xyz, plate xyz, GT phase, oracle action, текущие или будущие GT-метки gripper"),
    ("Выход", "EEF keyframe / историческое действие и вероятность закрытия gripper"),
    ("Развёртывание", "порог вероятности gripper threshold=0.5, затем преобразование в бинарную команду выполнения"),
]
md_table(["Категория", "Содержание"], rows)


| 类别 | 内容 |
| --- | --- |
| 允许 | agent/wrist 图像、语言、6D robot proprio、上一条策略自己执行过的 7D EEF/gripper 命令 |
| 禁止 | target xyz、plate xyz、GT phase、oracle action、当前或未来 GT gripper 标签 |
| 输出 | EEF keyframe/历史动作与 gripper close probability |
| 部署 | gripper probability threshold=0.5，再转成二值执行命令 |


У этого head входная граница совместима с VLA, но это дополнительно обученный визуальный и исторический управляющий head, поэтому в отчёте пишите `pi0 + learned head`, а не raw pi0. Он выучивает стадийные признаки из изображений и истории робота и не опирается на яркие маркеры или ручные координаты цели.


## Checkpoint 3: фактическая команда обучения visual/history head


In [5]:
import shlex

HEAD_SOURCE_REPO = os.environ.get("HEAD_SOURCE_REPO", "your_multiseed_fulltask_dataset")
HEAD_SOURCE_ROOT = Path(os.environ.get("HEAD_SOURCE_ROOT", DATA_ROOT / HEAD_SOURCE_REPO))
HEAD_SOURCE_JSONL = Path(os.environ.get("HEAD_SOURCE_JSONL", OUTPUT_ROOT / "collection_results.jsonl"))
HEAD_POLICY_PATH = Path(os.environ.get("HEAD_POLICY_PATH", MODEL_ROOT / "pi0_base" / "pretrained_model"))
HEAD_DIR = Path(os.environ.get("HEAD_DIR", MODEL_ROOT / "pi0_visual_history_head"))
FEATURE_CACHE = HEAD_DIR / "visual_features.npz"
DIRECT_HEAD = HEAD_DIR / "visual_keyframe_head.npz"
GRU_HEAD = HEAD_DIR / "visual_keyframe_gru_head.npz"
RUN_AUXILIARY_TRAIN = False

commands = [
    [
        sys.executable,
        str(TOPIC_ROOT / "code" / "pi0" / "train_pi0_visual_contact_head.py"),
        "--source", HEAD_SOURCE_REPO, str(HEAD_SOURCE_ROOT), str(HEAD_SOURCE_JSONL),
        "--policy-path", str(HEAD_POLICY_PATH),
        "--stats-repo-id", HEAD_SOURCE_REPO,
        "--stats-dataset-root", str(HEAD_SOURCE_ROOT),
        "--train-seeds", "21-32",
        "--val-seeds", "33-36",
        "--append-prev-action",
        "--feature-cache", str(FEATURE_CACHE),
        "--output", str(DIRECT_HEAD),
        "--summary-json", str(HEAD_DIR / "visual_keyframe_head_summary.json"),
    ],
    [
        sys.executable,
        str(TOPIC_ROOT / "code" / "pi0" / "train_pi0_visual_gripper_gru.py"),
        "--source", HEAD_SOURCE_REPO, str(HEAD_SOURCE_ROOT), str(HEAD_SOURCE_JSONL),
        "--feature-cache", str(FEATURE_CACHE),
        "--direct-head", str(DIRECT_HEAD),
        "--train-seeds", "21-32",
        "--val-seeds", "33-36",
        "--epochs", "300",
        "--transition-weight", "20",
        "--release-weight", "40",
        "--output", str(GRU_HEAD),
        "--summary-json", str(HEAD_DIR / "visual_keyframe_gru_summary.json"),
    ],
    [
        sys.executable,
        str(TOPIC_ROOT / "code" / "pi0" / "test_pi0_visual_gripper_gru_parity.py"),
        "--head", str(GRU_HEAD),
        "--steps", "32",
    ],
]

for command in commands:
    print("$", shlex.join(command))
if RUN_AUXILIARY_TRAIN:
    HEAD_DIR.mkdir(parents=True, exist_ok=True)
    for command in commands:
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("По умолчанию выводится только предпросмотр. Сначала замените пути к многопозиционным данным, JSONL коллектора и base policy, и только потом включите RUN_AUXILIARY_TRAIN.")


$ $NOTEBOOK_PYTHON $NOTEBOOK_TOPIC_ROOT/code/pi0/train_pi0_visual_contact_head.py --source your_multiseed_fulltask_dataset '$PROJECT_ROOT/your_multiseed_fulltask_dataset' '$OUTPUT_ROOT/collection_results.jsonl' --policy-path '$MODEL_ROOT/pi0_base/pretrained_model' --stats-repo-id your_multiseed_fulltask_dataset --stats-dataset-root '$PROJECT_ROOT/your_multiseed_fulltask_dataset' --train-seeds 21-32 --val-seeds 33-36 --append-prev-action --feature-cache '$MODEL_ROOT/pi0_visual_history_head/visual_features.npz' --output '$MODEL_ROOT/pi0_visual_history_head/visual_keyframe_head.npz' --summary-json '$MODEL_ROOT/pi0_visual_history_head/visual_keyframe_head_summary.json'
$ $NOTEBOOK_PYTHON $NOTEBOOK_TOPIC_ROOT/code/pi0/train_pi0_visual_gripper_gru.py --source your_multiseed_fulltask_dataset '$PROJECT_ROOT/your_multiseed_fulltask_dataset' '$OUTPUT_ROOT/collection_results.jsonl' --feature-cache '$MODEL_ROOT/pi0_visual_history_head/visual_features.npz' --direct-head '$MODEL_ROOT/pi0_visual_hist

## Checkpoint 4: формулировка языка тоже относится к протоколу развёртывания


In [6]:
rows = [
    (
        "Place the blue mug on the plate.",
        "0 active steps",
        "0/1",
        "короткий prompt выходит за пределы распределения обучения малой головы, prototype gate постоянно отклоняет передачу управления",
    ),
    (
        "Pick up the blue mug and place it on the plate.",
        "269 active steps",
        "1/1",
        "совпадает с инструкцией обучения; seed3 проходит final strict на 270 шагах",
    ),
]
md_table(["instruction", "learned head", "strict", "Явление"], rows)


| instruction | learned head | strict | 现象 |
| --- | --- | --- | --- |
| Place the blue mug on the plate. | 0 active steps | 0/1 | 短 prompt 超出小头训练分布，prototype gate 一直拒绝接管 |
| Pick up the blue mug and place it on the plate. | 269 active steps | 1/1 | 与训练指令一致；seed3 在 270 步通过 final strict |


Это не значит, что VLA никогда не может принимать синонимичные инструкции; просто auxiliary head, обученная на 4 коротких наборах данных, пока не обобщает перефразировки языка. При развёртывании сначала зафиксируйте prompt обучения и оценки; в следующем раунде добавьте аугментацию синонимичными инструкциями и отдельно протестируйте language paraphrase gate.


## Checkpoint 5: почему overall accuracy может обманывать


In [7]:
rows = [
    ("Старая линейная gripper head", "99.66%", "0/4", "кадров release очень мало, поэтому общая точность скрывает полный пропуск"),
    ("GRU sequence head", "96.8% balanced", "5/5", "порог подбирается отдельно для переходов close и release"),
]
md_table(["Модель", "Метрика валидации", "held-out release", "Интерпретация"], rows)


| 模型 | 验证指标 | held-out release | 解读 |
| --- | --- | --- | --- |
| 旧线性 gripper head | 99.66% | 0/4 | release 只有极少帧，总体准确率掩盖了全漏 |
| GRU sequence head | 96.8% balanced | 5/5 | 按 close/release transition 单独选阈值 |


## Checkpoint 6: воспроизведение итогового предиката успеха кодом


In [8]:
def final_strict_success(row):
    p = results["strict_predicate"]
    return bool(
        row["legacy_success"]
        and row["max_target_lift"] >= p["min_lift_m"]
        and row["max_lifted_run"] >= p["min_lift_steps"]
        and row["upright_cos"] >= p["min_upright_cos"]
        and row["plate_z_gap"] <= p["max_plate_z_gap_m"]
        and row["plate_xy_displacement"] <= p["max_plate_xy_displacement_m"]
        and row["stable_place_steps"] >= p["stable_place_steps"]
    )


known_success = {
    "legacy_success": True,
    "max_target_lift": 0.11,
    "max_lifted_run": 8,
    "upright_cos": 0.95,
    "plate_z_gap": 0.0448,
    "plate_xy_displacement": 0.0105,
    "stable_place_steps": 5,
}
print("Проходит ли пример итоговую строгую проверку:", final_strict_success(known_success))


示例是否通过最终严格判定： True


## Checkpoint 7: влияние ошибки seed на выводы


In [9]:
seed_bug = results["seed_bug"]
print("Старый код:", seed_bug["old_code"])
print("После исправления:", seed_bug["fixed_code"])
print("Влияние:", seed_bug["impact"])


旧代码： if seed != None: np.random.seed(seed=0)
修复后： if seed is not None: np.random.seed(seed=seed)
影响： 旧 30-seed 面板只改变策略采样随机性，环境始终是 seed 0，不能作为空间泛化证据


## Checkpoint 8: четырёхракурсное воспроизведение успеха

        ![Ключевые кадры четырёхракурсного воспроизведения успеха](../assets/pnp_four_view_strict_success_sequence.jpg)

        <video controls width="960" src="../assets/pnp_four_view_strict_success.mp4"></video>


## Checkpoint 9: план данных следующего раунда


In [10]:
plan = [
    ("Сбор", "после исправления seed собрать 30–50 полных успешных траекторий с разными положениями"),
    ("Разбиение", "по группам: красная и синяя чашка, область старта, contact miss, transport tip, release fail"),
    ("Обучение", "сначала защитить fixed-scene 6/12, затем обучить visual/history head; не добавляйте шаги raw pi0 вслепую"),
    ("Оценка", "fixed scene и random env раздельно; случайных окружений минимум 20–30"),
    ("Порог замены", "новая модель по тому же протоколу превосходит защищённый baseline, а видео без ложных срабатываний толчков и зависаний"),
]
md_table(["Этап", "Действие"], plan)


| 阶段 | 动作 |
| --- | --- |
| 采集 | 修正 seed 后采 30–50 条多位置完整成功轨迹 |
| 分层 | 按红/蓝杯、起点区域、contact miss、transport tip、release fail 分桶 |
| 训练 | 先保护 fixed-scene 6/12，再训练 visual/history head；不要继续盲加 raw pi0 steps |
| 评估 | fixed scene 与 random env 分开；随机环境至少 20–30 条 |
| 替换门槛 | 新模型同协议超过保护基线，且视频无推杯/悬空误判 |
